In [60]:
from facenet_pytorch import MTCNN,InceptionResnetV1
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
import torch
from torch.nn.functional import cosine_similarity

In [47]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Will Run models on {device}")

Will Run models on cpu


In [62]:
mtcnn = MTCNN(device=device,keep_all=True)
resnet = InceptionResnetV1(pretrained="vggface2").eval().to(device)

In [73]:
##Pipeline For authentication
def collate_fn(x):
    return x[0]
dataset = ImageFolder("test")
dataset.idx_to_class = {i:c for c,i in dataset.class_to_idx.items()}
loader = DataLoader(dataset,collate_fn=collate_fn,num_workers=0)


In [74]:
aligned = []
names = []
for x,y in loader:
    x_aligned,prob = mtcnn.forward(x,return_prob=True)
    for i in range(len(x_aligned)):
        t,p = x_aligned[i],prob[i]
        print("found face with prob: {}".format(p))
        aligned.append(t)
        names.append(dataset.idx_to_class[y])

found face with prob: 0.9996028542518616
found face with prob: 0.9998292922973633
found face with prob: 0.9999978542327881


In [75]:
names

['actual', 'actual', 'given']

In [76]:
aligned = torch.stack(aligned).to(device)
embeddings = resnet(aligned)
print(embeddings.shape)
similarity = cosine_similarity(embeddings[0],embeddings[1],dim=-1)
print(similarity)

torch.Size([3, 512])
tensor(0.0135, grad_fn=<SumBackward1>)


In [56]:
if similarity > 0.75:
    print("Allowed")
else:
    print("rejected")

Allowed


In [ ]:
## Folder called referance that stores the users image. 
# A photo is taken during inference and number of faces are detected 
# and compared against referance photo. 
#During inference if more than one face is detected, take similarity 
#scores with all images and the max_sim > threshold. 

In [77]:
from facenet_pytorch import MTCNN, InceptionResnetV1
import torch 
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from torch.nn.functional import cosine_similarity

In [79]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Currently running on {device}")

Currently running on cpu


In [88]:
mtcnn = MTCNN(device=device,keep_all=True)
resnet = InceptionResnetV1(pretrained="vggface2").eval().to(device)

In [82]:
def collate_fn(x):
    return x[-1]
dataset = ImageFolder('test')
dataset.idx_to_class = {i:c for c,i in dataset.class_to_idx.items()}
loader = DataLoader(dataset=dataset,collate_fn=collate_fn,num_workers=0)

In [85]:
dataset.class_to_idx

{'inference': 0, 'referance': 1}

In [89]:
inference = []
referance = []

for x,y in loader:
    faces_list,probs = mtcnn.forward(x,return_prob=True)
    class_name = dataset.idx_to_class[y]
    for i in range(len(faces_list)):
        face,prob = faces_list[i],probs[i]
        if class_name == "referance":
            referance.append(face)
        elif class_name == "inference":
            inference.append(face)
                
        print(f"Found face with prob: {prob}")

Found face with prob: 0.9996028542518616
Found face with prob: 0.9998292922973633
Found face with prob: 0.9999978542327881


In [94]:
referance = torch.stack(referance).to(device)
inference = torch.stack(inference).to(device)

In [ ]:
ref_emb = resnet.forward(referance)
ref_emb = ref_emb[-1]
inf_emb = resnet.forward(inference)

In [96]:
max_sim = -float('inf')
for emb in inf_emb:
    max_sim = max(cosine_similarity(emb,ref_emb,dim=-1),max_sim)

print(max_sim)
if max_sim > 0.75:
    print("Allow")
else:
    print("rejected")


tensor([0.6752], grad_fn=<SumBackward1>)
rejected
